# 02 MovieLens — Manipulación de DataFrames

**Clase 5 — Laboratorio 1**

## Objetivo
Construir y manipular DataFrames con PySpark: carga con schema explícito, transformaciones (`split`, `regexp_extract`, `withColumn`, `drop`) y comprensión de la ejecución *lazy* (transformaciones vs. acciones).

## Dataset
MovieLens: `ratings.csv`, `movies.csv`, `tags.csv` — almacenados en Unity Catalog Volumes.

---
> **Ejecuta *Restart & Run All* antes de entregar** para asegurarte de que el notebook corre limpio sin estado oculto.

Imports de PySpark necesarias para el notebook. En Free Edition la `SparkSession` ya está disponible como `spark` — no es necesario inicializarla.

In [0]:
# En Databricks Free Edition el objeto `spark` ya existe — no se necesita SparkSession.builder
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DecimalType
from pyspark.sql.functions import split, col, regexp_extract, from_unixtime

# Sesión

En Databricks Free Edition la `SparkSession` ya está disponible como `spark` — no es necesario llamar a `SparkSession.builder`. La sesión permite a todos los procesos involucrados compartir contexto.

In [0]:
#spark = SparkSession.builder.appName("movielens").getOrCreate()


# Carga de datos

Cargamos las tres tablas del dataset MovieLens: `ratings`, `movies` y `tags`. La primera celda hace una **lectura ingenua** (sin esquema ni header) a propósito, para mostrar qué produce Spark por defecto. Las siguientes celdas recargan cada tabla con esquema explícito y opciones correctas.

In [0]:
BASE= "/Volumes/big_data_2026/spark_examples/spark_data"
ratingsDf = spark.read.csv(f"{BASE}/ratings.csv")
moviesDf  = spark.read.csv(f"{BASE}/movies.csv")
tagsDf    = spark.read.csv(f"{BASE}/tags.csv")

Revisamos primero el dataframe de ratings

Si leemos un CSV sin schema ni header, ¿qué nombres y tipos tendrán las columnas? ¿Será la primera fila parte de los datos?

In [0]:
ratingsDf.head(2)

[Row(_c0='userId', _c1='movieId', _c2='rating', _c3='timestamp'),
 Row(_c0='1', _c1='1', _c2='4.0', _c3='964982703')]

Mostramos las primeras 2 filas de `ratingsDf` sin procesar: primera fila es el header, todos los tipos son `string`.

In [0]:
ratingsDf.show(2)

+------+-------+------+---------+
|   _c0|    _c1|   _c2|      _c3|
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
|     1|      1|   4.0|964982703|
+------+-------+------+---------+
only showing top 2 rows


Vemos que tiene header, y los tipos de datos, incluyendo el ultimo que es un timestamp. Podemos definir el schema:

In [0]:
ratings_schema = StructType(fields=[
    StructField("userId",   IntegerType(),              True), 
    StructField("movieId",  IntegerType(),              True),
    StructField("rating",   DecimalType(precision=2, scale=1), True),
    StructField("timestamp",LongType(),                 True)  # epoch en segundos; se convierte con from_unixtime()
])

Recargamos `ratingsDf` con el esquema explícito y `header=True`. Verificamos que los tipos son correctos con `head(2)`.

In [0]:
ratingsDf = spark.read\
    .option("header", True)\
    .option("dateFormat", "yyyyMMdd")\
    .schema(ratings_schema)\
    .csv(f"{BASE}/ratings.csv")
ratingsDf.head(2)

[Row(userId=1, movieId=1, rating=Decimal('4.0'), timestamp=964982703),
 Row(userId=1, movieId=3, rating=Decimal('4.0'), timestamp=964981247)]

Acabamos de definir `timestamp` como `LongType`. ¿Aparecerá como `long` o como `timestamp` en el esquema? ¿Y `rating` como `DecimalType(2,1)`?

Verificamos el esquema de `ratingsDf`: `userId` y `movieId` son `IntegerType`, `rating` es `DecimalType(2,1)` y `timestamp` es `LongType` (epoch).

In [0]:
ratingsDf.printSchema()

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: decimal(2,1) (nullable = true)
 |-- timestamp: long (nullable = true)



Cargamos `moviesDf` con su esquema explícito. La columna `genres` contiene valores separados por `|` que separaremos más adelante.

In [0]:
movies_schema = StructType(fields=[
    StructField("movieId", IntegerType(), True), 
    StructField("title",   StringType(),  True),
    StructField("genres",  StringType(),  True)
])

moviesDf = spark.read\
    .option("header", True)\
    .schema(movies_schema)\
    .csv(f"{BASE}/movies.csv")
moviesDf.printSchema()

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)



Primeras 5 filas de `moviesDf`: podemos ver la columna `genres` con valores `|`-delimitados y `title` con el año entre paréntesis.

In [0]:
moviesDf.show(5)

+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows


Notamos dos cosas:

1. En la columna `genres` vienen multiples valores separados por `|`
1. En la columna `title` viene tanto el nombre de la palicula como el año en que salió

Queremos separar esto.

In [0]:
# La función split separa una columna en un arreglo
moviesDf.select(split(col("genres"), "\\|").alias("genresSplit")).show()

+--------------------+
|         genresSplit|
+--------------------+
|[Adventure, Anima...|
|[Adventure, Child...|
|   [Comedy, Romance]|
|[Comedy, Drama, R...|
|            [Comedy]|
|[Action, Crime, T...|
|   [Comedy, Romance]|
|[Adventure, Child...|
|            [Action]|
|[Action, Adventur...|
|[Comedy, Drama, R...|
|    [Comedy, Horror]|
|[Adventure, Anima...|
|             [Drama]|
|[Action, Adventur...|
|      [Crime, Drama]|
|    [Drama, Romance]|
|            [Comedy]|
|            [Comedy]|
|[Action, Comedy, ...|
+--------------------+
only showing top 20 rows


O de otra forma:

Al agregar `genresSplit` con `split()`, ¿qué tipo tendrá esa columna en el esquema? ¿Desaparece `genres` automáticamente?

In [0]:
moviesDf.withColumn("genresSplit", split(moviesDf["genres"],"\|")).show()


+-------+--------------------+--------------------+--------------------+
|movieId|               title|              genres|         genresSplit|
+-------+--------------------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|[Adventure, Anima...|
|      2|      Jumanji (1995)|Adventure|Childre...|[Adventure, Child...|
|      3|Grumpier Old Men ...|      Comedy|Romance|   [Comedy, Romance]|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|[Comedy, Drama, R...|
|      5|Father of the Bri...|              Comedy|            [Comedy]|
|      6|         Heat (1995)|Action|Crime|Thri...|[Action, Crime, T...|
|      7|      Sabrina (1995)|      Comedy|Romance|   [Comedy, Romance]|
|      8| Tom and Huck (1995)|  Adventure|Children|[Adventure, Child...|
|      9| Sudden Death (1995)|              Action|            [Action]|
|     10|    GoldenEye (1995)|Action|Adventure|...|[Action, Adventur...|
|     11|American Presiden...|Comedy|Drama|Romance|

Creamos `moviesDfSplit` con la columna `genresSplit` (array de géneros) y mostramos el esquema: `genres` sigue presente, `genresSplit` es un `ArrayType(StringType)`.

In [0]:
moviesDfSplit = moviesDf.withColumn("genresSplit", split(moviesDf["genres"],"\|"))
moviesDfSplit.printSchema()

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- genresSplit: array (nullable = true)
 |    |-- element: string (containsNull = false)



No es necesario mantener ambas columnas, entonces es posible eliminar la columna `genres`

In [0]:
# Llamamos drop sin reasignar — ¿desaparecerá genres?
moviesDfSplit.drop("genres")

DataFrame[movieId: int, title: string, genresSplit: array<string>]

Llamamos `moviesDfSplit.drop("genres")`. ejecuta `printSchema()` abajo — ¿seguirá apareciendo `genres`? ¿Por qué?

Ahora revisamos el dataframe

In [0]:
moviesDfSplit.printSchema()

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- genresSplit: array (nullable = true)
 |    |-- element: string (containsNull = false)



**Inmutabilidad:** `genres` sigue apareciendo — el `drop()` devolvió un *nuevo* DataFrame pero no lo asignamos, así que el cambio se perdió. Los DataFrames de Spark son **inmutables**: cada transformación crea un nuevo DataFrame. Para conservar el resultado hay que reasignar:

In [0]:
moviesDfSplit = moviesDfSplit.drop("genres")
moviesDfSplit.printSchema()

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genresSplit: array (nullable = true)
 |    |-- element: string (containsNull = false)



Ahora sucede lo mismo con la columna title. Podemos extrar el nombre de la película y el año y crear columnas especificas para cada dato. Usamos expresiones regulares con la funcion [regexp_extract](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.regexp_extract.html#pyspark.sql.functions.regexp_extract)

In [0]:
moviesDfSplit.withColumn("year", regexp_extract(col("title"), "^.+\(([0-9]+)\)$", 1)).show()

+-------+--------------------+--------------------+----+
|movieId|               title|         genresSplit|year|
+-------+--------------------+--------------------+----+
|      1|    Toy Story (1995)|[Adventure, Anima...|1995|
|      2|      Jumanji (1995)|[Adventure, Child...|1995|
|      3|Grumpier Old Men ...|   [Comedy, Romance]|1995|
|      4|Waiting to Exhale...|[Comedy, Drama, R...|1995|
|      5|Father of the Bri...|            [Comedy]|1995|
|      6|         Heat (1995)|[Action, Crime, T...|1995|
|      7|      Sabrina (1995)|   [Comedy, Romance]|1995|
|      8| Tom and Huck (1995)|[Adventure, Child...|1995|
|      9| Sudden Death (1995)|            [Action]|1995|
|     10|    GoldenEye (1995)|[Action, Adventur...|1995|
|     11|American Presiden...|[Comedy, Drama, R...|1995|
|     12|Dracula: Dead and...|    [Comedy, Horror]|1995|
|     13|        Balto (1995)|[Adventure, Anima...|1995|
|     14|        Nixon (1995)|             [Drama]|1995|
|     15|Cutthroat Island ...|[

`regexp_extract` devuelve la cadena capturada. ¿Qué tipo tendrá `year` antes de hacer el cast? ¿Y después de `.try_cast(IntegerType())`? ¿Qué ocurre si el título no tiene año entre paréntesis?

Revisamos el schema y vemos que es de tipo string, deberia ser numerico.

In [0]:
moviesDfSplit.withColumn("year", regexp_extract(col("title"), "^.+\(([0-9]+)\)$", 1)).printSchema()

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genresSplit: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- year: string (nullable = true)



Extraemos el año con `regexp_extract` y lo casteamos a `IntegerType()`. La función `cast()` convierte el tipo de la columna resultante.

In [0]:
moviesDfSplit.withColumn("year", regexp_extract(col("title"), "^.+\(([0-9]+)\)$", 1).try_cast(IntegerType())).show()

+-------+--------------------+--------------------+----+
|movieId|               title|         genresSplit|year|
+-------+--------------------+--------------------+----+
|      1|    Toy Story (1995)|[Adventure, Anima...|1995|
|      2|      Jumanji (1995)|[Adventure, Child...|1995|
|      3|Grumpier Old Men ...|   [Comedy, Romance]|1995|
|      4|Waiting to Exhale...|[Comedy, Drama, R...|1995|
|      5|Father of the Bri...|            [Comedy]|1995|
|      6|         Heat (1995)|[Action, Crime, T...|1995|
|      7|      Sabrina (1995)|   [Comedy, Romance]|1995|
|      8| Tom and Huck (1995)|[Adventure, Child...|1995|
|      9| Sudden Death (1995)|            [Action]|1995|
|     10|    GoldenEye (1995)|[Action, Adventur...|1995|
|     11|American Presiden...|[Comedy, Drama, R...|1995|
|     12|Dracula: Dead and...|    [Comedy, Horror]|1995|
|     13|        Balto (1995)|[Adventure, Anima...|1995|
|     14|        Nixon (1995)|             [Drama]|1995|
|     15|Cutthroat Island ...|[

Verificamos el esquema: la columna `year` es ahora `IntegerType` en lugar de `StringType`.

In [0]:
moviesDfSplit.withColumn("year", regexp_extract(col("title"), "^.+\(([0-9]+)\)$", 1).try_cast(IntegerType())).printSchema()

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genresSplit: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- year: integer (nullable = true)



Asignamos el año como `IntegerType` de vuelta a `moviesDfSplit`. Los DataFrames son inmutables — hay que reasignar el resultado de la transformación.

In [0]:
moviesDfSplit = (moviesDfSplit
                 .withColumn("year",
                             regexp_extract(col("title"), "^.+\(([0-9]+)\)$", 1)
                             .try_cast(IntegerType())))

Existen muchas funciones similares que se pueden utilizar, la documentación está en [Spark SQL Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)

In [0]:
moviesDfSplit.show(5)

+-------+--------------------+--------------------+----+
|movieId|               title|         genresSplit|year|
+-------+--------------------+--------------------+----+
|      1|    Toy Story (1995)|[Adventure, Anima...|1995|
|      2|      Jumanji (1995)|[Adventure, Child...|1995|
|      3|Grumpier Old Men ...|   [Comedy, Romance]|1995|
|      4|Waiting to Exhale...|[Comedy, Drama, R...|1995|
|      5|Father of the Bri...|            [Comedy]|1995|
+-------+--------------------+--------------------+----+
only showing top 5 rows


Hacemos lo mismo para obtener el titulo

In [0]:
moviesDfSplit = (moviesDfSplit
                 .withColumn("title_temp",
                             regexp_extract(col("title"), "^(.+?) \([0-9]+\)$", 1))
                 .drop("title")
                 .withColumnRenamed("title_temp", "title"))
moviesDfSplit.printSchema()

root
 |-- movieId: integer (nullable = true)
 |-- genresSplit: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- year: integer (nullable = true)
 |-- title: string (nullable = true)



Verificamos `moviesDfSplit` con las columnas `genresSplit`, `year` y `title` ya limpias y correctamente tipadas.

In [0]:
moviesDfSplit.explain()

== Physical Plan ==
PhotonResultStage
+- PhotonColumnarToRow
   +- PhotonProject [movieId#11293, split(genres#11295, \|, -1) AS genresSplit#11298, try_cast(regexp_extract(title#11294, ^.+\(([0-9]+)\)$, 1) as int) AS year#11300, regexp_extract(title#11294, ^(.+?) \([0-9]+\)$, 1) AS title#11303]
      +- PhotonRowToColumnar
         +- FileScan csv [movieId#11293,title#11294,genres#11295] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/big_data_2026/spark_examples/spark_data/movies.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<movieId:int,title:string,genres:string>


== Photon Explanation ==
The query is fully supported by Photon.


In [0]:
moviesDfSplit.show(5)

+-------+--------------------+----+--------------------+
|movieId|         genresSplit|year|               title|
+-------+--------------------+----+--------------------+
|      1|[Adventure, Anima...|1995|           Toy Story|
|      2|[Adventure, Child...|1995|             Jumanji|
|      3|   [Comedy, Romance]|1995|    Grumpier Old Men|
|      4|[Comedy, Drama, R...|1995|   Waiting to Exhale|
|      5|            [Comedy]|1995|Father of the Bri...|
+-------+--------------------+----+--------------------+
only showing top 5 rows


Por ultimo hacemos lo mismo con la tabla de tags

In [0]:
tagsDf.show()

+------+-------+--------------------+----------+
|   _c0|    _c1|                 _c2|       _c3|
+------+-------+--------------------+----------+
|userId|movieId|                 tag| timestamp|
|    10|    260|        good vs evil|1430666558|
|    10|    260|       Harrison Ford|1430666505|
|    10|    260|              sci-fi|1430666538|
|    14|   1221|           Al Pacino|1311600756|
|    14|   1221|               mafia|1311600746|
|    14|  58559|         Atmospheric|1311530439|
|    14|  58559|              Batman|1311530391|
|    14|  58559|          comic book|1311530398|
|    14|  58559|                dark|1311530428|
|    14|  58559|        Heath Ledger|1311530404|
|    14|  58559|        imdb top 250|1311530451|
|    14|  58559|       Michael Caine|1311530407|
|    14|  58559|      Morgan Freeman|1311530400|
|    14|  58559|Oscar (Best Suppo...|1311530432|
|    14|  58559|          psychology|1311530417|
|    14|  58559|           superhero|1311530388|
|    14|  58559|    

Cargamos `tagsDf` con su esquema explícito. El campo `timestamp` es `LongType` (epoch); lo convertiremos a fecha con `from_unixtime()`.

In [0]:
tags_schema = StructType(fields=[    
    StructField("userId",    IntegerType(), True), 
    StructField("movieId",   IntegerType(), True), 
    StructField("tag",       StringType(),  True),
    StructField("timestamp", LongType(),    True)  # epoch en segundos
])

tagsDf = spark.read\
    .option("header", True)\
    .schema(tags_schema)\
    .csv(f"{BASE}/tags.csv")

tagsDf.show()

+------+-------+--------------------+----------+
|userId|movieId|                 tag| timestamp|
+------+-------+--------------------+----------+
|    10|    260|        good vs evil|1430666558|
|    10|    260|       Harrison Ford|1430666505|
|    10|    260|              sci-fi|1430666538|
|    14|   1221|           Al Pacino|1311600756|
|    14|   1221|               mafia|1311600746|
|    14|  58559|         Atmospheric|1311530439|
|    14|  58559|              Batman|1311530391|
|    14|  58559|          comic book|1311530398|
|    14|  58559|                dark|1311530428|
|    14|  58559|        Heath Ledger|1311530404|
|    14|  58559|        imdb top 250|1311530451|
|    14|  58559|       Michael Caine|1311530407|
|    14|  58559|      Morgan Freeman|1311530400|
|    14|  58559|Oscar (Best Suppo...|1311530432|
|    14|  58559|          psychology|1311530417|
|    14|  58559|           superhero|1311530388|
|    14|  58559|           vigilante|1311530423|
|    14|  58559|    

Verificamos el esquema de `tagsDf`: `userId`, `movieId` como `IntegerType`, `tag` como `StringType` y `timestamp` como `LongType`.

In [0]:
tagsDf.printSchema()

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- tag: string (nullable = true)
 |-- timestamp: long (nullable = true)



In [0]:
tagsDf.describe().show()

+-------+------------------+-----------------+--------------+--------------------+
|summary|            userId|          movieId|           tag|           timestamp|
+-------+------------------+-----------------+--------------+--------------------+
|  count|           2328315|          2328315|       2328315|             2328312|
|   mean|177666.48688600984|70552.80300088262|      Infinity| 1.522918579022266E9|
| stddev| 82856.68529127882|74024.26926168487|           NaN|1.3085465405347005E8|
|    min|                10|                1|    The Asylum|          1135429210|
|    max|            330947|           288955|    카운트다운|          1689841404|
+-------+------------------+-----------------+--------------+--------------------+



Finalmente, convertimos el timestamp en una fecha:

In [0]:
tagsDf = spark.read\
    .option("header", True)\
    .schema(tags_schema)\
    .csv(f"{BASE}/tags.csv")

tagsDf = tagsDf\
    .withColumn("date", from_unixtime("timestamp", "yyyyMMdd"))\
    .drop("timestamp")

tagsDf.show()

+------+-------+--------------------+--------+
|userId|movieId|                 tag|    date|
+------+-------+--------------------+--------+
|    10|    260|        good vs evil|20150503|
|    10|    260|       Harrison Ford|20150503|
|    10|    260|              sci-fi|20150503|
|    14|   1221|           Al Pacino|20110725|
|    14|   1221|               mafia|20110725|
|    14|  58559|         Atmospheric|20110724|
|    14|  58559|              Batman|20110724|
|    14|  58559|          comic book|20110724|
|    14|  58559|                dark|20110724|
|    14|  58559|        Heath Ledger|20110724|
|    14|  58559|        imdb top 250|20110724|
|    14|  58559|       Michael Caine|20110724|
|    14|  58559|      Morgan Freeman|20110724|
|    14|  58559|Oscar (Best Suppo...|20110724|
|    14|  58559|          psychology|20110724|
|    14|  58559|           superhero|20110724|
|    14|  58559|           vigilante|20110724|
|    14|  58559|            violence|20110724|
|    16|  571

---
## Ejercicios — Tarea Corta #3

Completa los 10 ejercicios usando los dataframes de este notebook. 

### Ejercicio 1 — Filtrar por año (transformación: filter)

Filtra `moviesDfSplit` para mostrar **solo las películas estrenadas a partir del año 2000**
(`year >= 2000`). Muestra las primeras 10 filas.

In [0]:
movies_2000 = moviesDfSplit.filter(col("year") >= 2000).limit(10)
display(movies_2000)

movieId,genresSplit,year,title
2769,"List(Crime, Drama)",2000,"Yards, The"
3177,List(Comedy),2000,Next Friday
3190,"List(Adventure, Sci-Fi, Thriller)",2000,Supernova
3225,"List(Comedy, Romance)",2000,Down to You
3228,List(Comedy),2000,Wirey Spindell
3239,List(Comedy),2000,Isn't She Great?
3273,"List(Comedy, Horror, Mystery, Thriller)",2000,Scream 3
3275,"List(Action, Crime, Drama, Thriller)",2000,"Boondock Saints, The"
3276,List(Comedy),2000,Gun Shy
3279,"List(Action, Drama)",2000,Knockout


### Ejercicio 2 — Seleccionar columnas (transformación: select)

A partir de `moviesDfSplit`, selecciona **únicamente** las columnas `title`, `year` y
`genresSplit`. Muestra 10 filas sin truncar el contenido del array.

*Nota:* `.show(10, truncate=False)`. :)

In [0]:
movies_selected = moviesDfSplit.select("title", "year", "genresSplit")
movies_selected.show(10, truncate=False)

+---------------------------+----+-------------------------------------------------+
|title                      |year|genresSplit                                      |
+---------------------------+----+-------------------------------------------------+
|Toy Story                  |1995|[Adventure, Animation, Children, Comedy, Fantasy]|
|Jumanji                    |1995|[Adventure, Children, Fantasy]                   |
|Grumpier Old Men           |1995|[Comedy, Romance]                                |
|Waiting to Exhale          |1995|[Comedy, Drama, Romance]                         |
|Father of the Bride Part II|1995|[Comedy]                                         |
|Heat                       |1995|[Action, Crime, Thriller]                        |
|Sabrina                    |1995|[Comedy, Romance]                                |
|Tom and Huck               |1995|[Adventure, Children]                            |
|Sudden Death               |1995|[Action]                       

### Ejercicio 3 — Añadir una columna calculada (transformación: withColumn)

Crea una columna nueva llamada `decada` que contenga la década de la película
(por ejemplo, 1994 → 1990). Asígnala a un nuevo DataFrame `moviesConDecada` y muestra
`title`, `year` y `decada`.

*Nota:* Recuerda que los DataFrames son **inmutables**

In [0]:
moviesConDecada = moviesDfSplit.withColumn("decada", (col("year") / 10).cast("int") * 10)
moviesConDecada.select("title", "year", "decada").show(10, truncate=False)

+---------------------------+----+------+
|title                      |year|decada|
+---------------------------+----+------+
|Toy Story                  |1995|1990  |
|Jumanji                    |1995|1990  |
|Grumpier Old Men           |1995|1990  |
|Waiting to Exhale          |1995|1990  |
|Father of the Bride Part II|1995|1990  |
|Heat                       |1995|1990  |
|Sabrina                    |1995|1990  |
|Tom and Huck               |1995|1990  |
|Sudden Death               |1995|1990  |
|GoldenEye                  |1995|1990  |
+---------------------------+----+------+
only showing top 10 rows


### Ejercicio 4 — Películas por año, ordenadas (groupBy + count + orderBy)

Cuenta **cuántas películas hay por año** en `moviesDfSplit` y ordena el resultado de
**mayor a menor** cantidad de películas. Muestra los 10 años con más estrenos.

*Nota:* Importar `desc` y `count`/`avg` de `pyspark.sql.functions`.

In [0]:
from pyspark.sql.functions import desc, count

movies_per_year = moviesDfSplit.groupBy("year").count().orderBy(desc("count"))
movies_per_year.show(10)

+----+-----+
|year|count|
+----+-----+
|2017| 3231|
|2018| 3170|
|2016| 3143|
|2019| 3055|
|2015| 3044|
|2014| 2871|
|2020| 2628|
|2013| 2598|
|2012| 2364|
|2021| 2296|
+----+-----+
only showing top 10 rows


### Ejercicio 5 — Calificación promedio por película (agregaciones)

Usando `ratingsDf`, calcula para **cada `movieId`**: el número de calificaciones
(`num_calificaciones`) y la calificación promedio (`rating_promedio`). Filtrar
las películas que tengan **al menos 50 calificaciones** y ordénalas por `rating_promedio`
descendente. Muestra las 10 primeras.

*Nota:* `.groupBy("movieId").agg(count("*").alias("num_calificaciones"), avg("rating").alias("rating_promedio"))` :)

In [0]:
from pyspark.sql.functions import avg

resultado_ratings = ratingsDf.groupBy("movieId").agg(
    count("*").alias("num_calificaciones"),
    avg("rating").alias("rating_promedio")
).filter("num_calificaciones >= 50").orderBy(desc("rating_promedio"))

resultado_ratings.show(10)

+-------+------------------+---------------+
|movieId|num_calificaciones|rating_promedio|
+-------+------------------+---------------+
|    318|               317|        4.42902|
|    858|               192|        4.28906|
|   2959|               218|        4.27294|
|   1276|                57|        4.27193|
|    750|                97|        4.26804|
|    904|                84|        4.26190|
|   1221|               129|        4.25969|
|  48516|               107|        4.25234|
|   1213|               126|        4.25000|
|    912|               100|        4.24000|
+-------+------------------+---------------+
only showing top 10 rows


### Ejercicio 6 — Unir calificaciones con títulos (join)

Une `ratingsDf` con `moviesDfSplit` por la columna `movieId` (inner join) para obtener un
DataFrame con el `título` de cada película junto a su `rating`. Muestra 5 filas con las
columnas `title`, `rating` y `userId`.

*Nota:* `ratingsDf.join(moviesDfSplit, on="movieId", how="inner")`.

In [0]:
joined_df = ratingsDf.join(moviesDfSplit, on="movieId", how="inner")
joined_df.select("title", "rating", "userId").show(5)

+--------------------+------+------+
|               title|rating|userId|
+--------------------+------+------+
|           Toy Story|   5.0|   610|
|             Jumanji|   2.0|   608|
|    Grumpier Old Men|   2.0|   608|
|   Waiting to Exhale|   1.5|   600|
|Father of the Bri...|   3.0|   604|
+--------------------+------+------+
only showing top 5 rows


### Ejercicio 7 — Trabajar con el array de genres (explode)

La columna `genresSplit` es un **array**. Usa `explode` para generar **una fila por cada
género** de cada película, y luego cuenta **cuántas películas hay por género**, ordenado de
mayor a menor. Muestra los 10 géneros más frecuentes.

*Nota:* crea una columna `genre` con `explode(col("genresSplit"))`, luego puede usar group by genre y ordenar por el conteo

In [0]:
from pyspark.sql.functions import explode, col, desc

genres_df = moviesDfSplit.withColumn("genre", explode(col("genresSplit")))
genres_counted = genres_df.groupBy("genre").count().orderBy(desc("count"))

genres_counted.show(10)

+------------------+-----+
|             genre|count|
+------------------+-----+
|             Drama|33681|
|            Comedy|22829|
|          Thriller|11675|
|           Romance|10172|
|            Action| 9563|
|       Documentary| 9283|
|            Horror| 8570|
|(no genres listed)| 7060|
|             Crime| 6917|
|         Adventure| 5349|
+------------------+-----+
only showing top 10 rows


### Ejercicio 8 — Agrupar columnas en un struct (struct)

Crea una columna `info` de tipo **struct** que agrupe `title`, `year` y `genresSplit` en un
solo campo anidado dentro de `moviesDfSplit`. Muestra `movieId` e `info` (sin truncar) para
5 películas.

In [0]:
from pyspark.sql.functions import struct

movies_struct = moviesDfSplit.withColumn("info", struct("title", "year", "genresSplit"))
movies_struct.select("movieId", "info").show(5, truncate=False)

+-------+--------------------------------------------------------------------+
|movieId|info                                                                |
+-------+--------------------------------------------------------------------+
|1      |{Toy Story, 1995, [Adventure, Animation, Children, Comedy, Fantasy]}|
|2      |{Jumanji, 1995, [Adventure, Children, Fantasy]}                     |
|3      |{Grumpier Old Men, 1995, [Comedy, Romance]}                         |
|4      |{Waiting to Exhale, 1995, [Comedy, Drama, Romance]}                 |
|5      |{Father of the Bride Part II, 1995, [Comedy]}                       |
+-------+--------------------------------------------------------------------+
only showing top 5 rows


### Ejercicio 9 — Top 3 calificaciones por usuario (función de ventana)

Para cada usuario, queremos sus **3 calificaciones más recientes**. Usa una función de
ventana particionada por `userId` y ordenada por `timestamp` descendente; numera las filas
con `row_number()` y quédate con las que tengan número ≤ 3. Muestra `userId`, `movieId`,
`rating` y el número de fila para 20 filas.

*Nota:* la partición debe ser una columna de **alta cardinalidad** (`userId`), no de baja
cardinalidad (`rating`), para repartir bien el trabajo.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

windowSpec = Window.partitionBy("userId").orderBy(col("timestamp").desc())

top3_ratings = ratingsDf.withColumn("rn", row_number().over(windowSpec)).filter(col("rn") <= 3)

top3_ratings.select("userId", "movieId", "rating", "rn").show(20)

+------+-------+------+---+
|userId|movieId|rating| rn|
+------+-------+------+---+
|     1|   2492|   4.0|  1|
|     1|   2012|   4.0|  2|
|     1|   2478|   4.0|  3|
|     2|  80489|   4.5|  1|
|     2| 114060|   2.0|  2|
|     2| 122882|   5.0|  3|
|     3|   2424|   0.5|  1|
|     3|   5048|   0.5|  2|
|     3|    527|   0.5|  3|
|     4|   4246|   4.0|  1|
|     4|   4741|   3.0|  2|
|     4|   4896|   4.0|  3|
|     5|    247|   5.0|  1|
|     5|    300|   3.0|  2|
|     5|    474|   4.0|  3|
|     6|    780|   5.0|  1|
|     6|   1061|   4.0|  2|
|     6|    979|   3.0|  3|
|     7|  49286|   0.5|  1|
|     7|  49278|   3.5|  2|
+------+-------+------+---+
only showing top 20 rows


### Ejercicio 10 — La misma pregunta con Spark SQL

Toma el DataFrame combinado del Ejercicio 6 (`ratingsDf` ⋈ `moviesDfSplit`), regístralo como
vista temporal `peliculas` y responde con **Spark SQL**: la **calificación promedio por
`title`**, solo para títulos con **más de 100 calificaciones**, ordenado de mayor a menor
promedio (top 10).

Además, **antes** de llamar a `.show()`, asigna la consulta a una variable y ejecuta
`.explain()` sobre ella. Observa que `explain()` muestra el plan **sin** procesar los datos:
la ejecución real ocurre solo cuando llamas a la acción `.show()`.

In [0]:
joined_df.createOrReplaceTempView("peliculas")

consulta_sql = spark.sql("""
    SELECT title, AVG(rating) AS rating_promedio
    FROM peliculas
    GROUP BY title
    HAVING COUNT(*) > 100
    ORDER BY rating_promedio DESC
""")

consulta_sql.explain()
consulta_sql.show(10)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonSort [rating_promedio#12496 DESC NULLS LAST]
         +- PhotonShuffleExchangeSource
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#8584]
               +- PhotonShuffleExchangeSink rangepartitioning(rating_promedio#12496 DESC NULLS LAST, 16)
                  +- PhotonProject [title#12488, rating_promedio#12496]
                     +- PhotonFilter (count(1)#12499L > 100)
                        +- PhotonGroupingAgg(keys=[title#12488], functions=[finalmerge_avg(merge sum#12502, count#12503L) AS avg(unscaledvalue(rating))#12497, finalmerge_count(merge count#12505L) AS count(1)#12498L])
                           +- PhotonShuffleExchangeSource
                              +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#8574]
                                 +- PhotonShuffleExchangeSink hashpartitioning(title#12488, 16)
          

---
> **Recuerda:** ejecuta *Run->Clear->Clear all cell outptus*  & *Run->Run and Debug->Run All* antes de entregar para verificar que el notebook corre limpio de principio a fin.